In [9]:
import os
import numpy as np
import pandas as pd
import glob
import random
from scipy.stats import pearsonr

# ==========================================
# 1. CONFIGURATION
# ==========================================
RAW_DATA_PATH = r"../../data/raw/multi_class_pothole"
MANUAL_DATA_PATH = r"../../data/raw/binary_manual_collection/Yashobhoomi.csv"
OUTPUT_DIR = r"../../data/combined/multi_class_pothole/"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

GRAVITY = 9.80665
TARGET_FREQ = 10
WINDOW_SIZE = 20
STRIDE = 5
TEST_RATIO = 0.2 

# File Rules: Maps filename to [(start, end, label)]
FILE_RULES = {
    "both bad (2).csv":        [(63, -1, 4)],
    "both bad (3).csv":        [(22, 420, 4), (608, -1, 4)],
    "both bad.csv":            [(0, -1, 4)],
    "both medium (2).csv":     [(270, -1, 3)],
    "Both medium.csv":         [(250, 450, 3)],
    "both small.csv":          [(340, -1, 2)],
    "both vibration.csv":      [(0, -1, 1)],
    "left medium.csv":         [(990, -1, 3)],
    "left_medium_right.csv":   [(140, 420, 3)],
    "right bad.csv":           [(0, -1, 4)],
    "right small (2).csv":     [(0, -1, 2)],
    "right small.csv":         [(0, 900, 0), (900, 1120, 2)],
    "vibrations_small_medium.csv": [(0, 520, 1), (520, 910, 2), (910, -1, 4)]
}

GOOD_SEGMENTS_COORDS = [
    (28.408258, 76.927417, 28.405190, 76.919277), 
    (28.517245, 77.022301, 28.463858, 76.968626)  
]

# ==========================================
# 2. CORE UTILITY FUNCTIONS
# ==========================================
def get_segment_indices(df, lat1, lon1, lat2, lon2):
    """Finds row indices closest to specified GPS coordinates."""
    dist_start = (df['latitude'] - lat1)**2 + (df['longitude'] - lon1)**2
    dist_end = (df['latitude'] - lat2)**2 + (df['longitude'] - lon2)**2
    return sorted([dist_start.idxmin(), dist_end.idxmin()])

def generate_synthetic_speed(length, base_speed_type):
    """Generates a noisy speed profile."""
    if base_speed_type == 'fast': target = np.random.uniform(10, 20)
    elif base_speed_type == 'slow': target = np.random.uniform(2, 8)
    else: target = np.random.uniform(2, 20)
    
    noise = np.random.normal(0, 0.1, length)
    speed_profile = np.cumsum(noise) + target
    return np.clip(speed_profile, 0.5, 30)

def smooth_transition(arr1, arr2, overlap=5):
    """Blends two sensor arrays to prevent sudden jerks at stitching points."""
    if len(arr1) < overlap or len(arr2) < overlap: 
        return np.vstack([arr1, arr2])
    mid = (arr1[-overlap:] + arr2[:overlap]) / 2 
    return np.vstack([arr1[:-overlap], mid, arr2[overlap:]])

# ==========================================
# 3. UPDATED DATA LOADING (MIXED LABEL 0)
# ==========================================
def load_and_split_pools():
    """
    Groups files by label for stratified pothole splitting, 
    but MIXES Label 0 (Good Road) chunks across Train and Test pools.
    """
    random.seed(42)
    
    # --- PART A: Stratified Split for Potholes (Labels 1-4) ---
    # Keep these strictly separated by file to prevent leakage
    label_groups = {1: [], 2: [], 3: [], 4: []}
    for filename, rules in FILE_RULES.items():
        max_label = max([r[2] for r in rules])
        if max_label > 0:
            label_groups[max_label].append(filename)

    test_files, train_files = [], []
    for lbl, files in label_groups.items():
        random.shuffle(files)
        # Ensure at least one file from each class goes to Test
        test_files.append(files[0])
        train_files.extend(files[1:])

    print(f"--- Stratified Split Complete ---")
    print(f"Test Pothole Files: {test_files}\n")

    pool_train, pool_test = [], []
    all_available_csvs = glob.glob(os.path.join(RAW_DATA_PATH, "*.csv"))
    
    # Process Pothole CSVs
    for fname in all_available_csvs:
        basename = os.path.basename(fname)
        if basename not in FILE_RULES: continue
        
        df = pd.read_csv(fname)
        if 'az' in df.columns: df['az'] += GRAVITY
        
        target_pool = pool_test if basename in test_files else pool_train
        
        for start, end, label in FILE_RULES[basename]:
            segment = df.iloc[start:] if end == -1 else df.iloc[start:end]
            segment = segment.iloc[::10] 
            data = segment[['ax', 'ay', 'az', 'wx', 'wy', 'wz']].values
            if len(data) >= WINDOW_SIZE:
                target_pool.append({'data': data, 'label': label})

    # --- PART B: Mix & Shuffle Yashobhoomi (Label 0) ---
    if os.path.exists(MANUAL_DATA_PATH):
        df_yash = pd.read_csv(MANUAL_DATA_PATH)
        cols = ['ax', 'ay', 'az', 'wx', 'wy', 'wz']
        
        good_road_chunks = []
        
        # 1. Extract BOTH segments
        for coords in GOOD_SEGMENTS_COORDS:
            s, e = get_segment_indices(df_yash, *coords)
            full_segment = df_yash.iloc[s:e][cols].values
            
            # 2. Chop into 5-second chunks (50 samples)
            # This allows us to shuffle them thoroughly
            chunk_size = 50 
            for i in range(0, len(full_segment) - chunk_size, chunk_size):
                chunk = full_segment[i : i + chunk_size]
                good_road_chunks.append(chunk)

        # 3. Shuffle all chunks together
        random.shuffle(good_road_chunks)
        
        # 4. Split based on TEST_RATIO
        split_idx = int(len(good_road_chunks) * (1 - TEST_RATIO))
        train_chunks = good_road_chunks[:split_idx]
        test_chunks = good_road_chunks[split_idx:]
        
        # 5. Add to pools
        for chunk in train_chunks:
            pool_train.append({'data': chunk, 'label': 0})
            
        for chunk in test_chunks:
            pool_test.append({'data': chunk, 'label': 0})

        print(f"Label 0 Mixing: {len(train_chunks)} chunks in Train, {len(test_chunks)} chunks in Test.")

    return pool_train, pool_test
# ==========================================
# 4. SYNTHESIS ENGINE
# ==========================================
def synthesize_windows(pool, num_scenarios, output_filename):
    """Stitches segments together and generates windowed CSV data."""
    output_rows = []
    window_counter = 0
    
    has_good = any(p['label'] == 0 for p in pool)
    has_bad = any(p['label'] > 0 for p in pool)
    
    if not pool:
        print(f"Skipping {output_filename}: Pool empty.")
        return
        
    print(f"Synthesizing {num_scenarios} scenarios for {output_filename}...")
    
    for _ in range(num_scenarios):
        if has_good and has_bad:
            seg_a = random.choice([p for p in pool if p['label'] == 0])
            seg_b = random.choice([p for p in pool if p['label'] > 0])
        else:
            seg_a, seg_b = random.choice(pool), random.choice(pool)

        chunk_len = np.random.randint(30, 60)
        idx_a = np.random.randint(0, max(1, len(seg_a['data'])-chunk_len))
        idx_b = np.random.randint(0, max(1, len(seg_b['data'])-chunk_len))
        
        clip_a = seg_a['data'][idx_a : idx_a+chunk_len]
        clip_b = seg_b['data'][idx_b : idx_b+chunk_len]
        
        raw_seq = smooth_transition(clip_a, clip_b)
        
        # Approximate labeling for the combined sequence
        combined_labels = np.concatenate([np.zeros(len(clip_a)), np.full(len(clip_b), seg_b['label'])])
        if len(combined_labels) > len(raw_seq): combined_labels = combined_labels[:len(raw_seq)]
        else: combined_labels = np.pad(combined_labels, (0, len(raw_seq)-len(combined_labels)), 'edge')

        spd_type = np.random.choice(['fast', 'slow', 'random'])
        speed_col = generate_synthetic_speed(len(raw_seq), spd_type)
        
        num_windows = (len(raw_seq) - WINDOW_SIZE) // STRIDE
        for i in range(num_windows):
            start = i * STRIDE
            end = start + WINDOW_SIZE
            win_data, win_speed = raw_seq[start:end], speed_col[start:end]
            win_label = int(np.max(combined_labels[start:end]))
            
            for step in range(WINDOW_SIZE):
                output_rows.append({
                    'window_id': window_counter, 'step': step,
                    'ax': win_data[step, 0], 'ay': win_data[step, 1], 'az': win_data[step, 2],
                    'wx': win_data[step, 3], 'wy': win_data[step, 4], 'wz': win_data[step, 5],
                    'speed': win_speed[step], 'label': win_label
                })
            window_counter += 1

    pd.DataFrame(output_rows).to_csv(output_filename, index=False)
    print(f"Successfully saved to {output_filename}")

# ==========================================
# 5. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    train_pool, test_pool = load_and_split_pools()
    
    synthesize_windows(train_pool, 2000, os.path.join(OUTPUT_DIR, "train.csv"))
    synthesize_windows(test_pool, 500, os.path.join(OUTPUT_DIR, "test.csv"))
    print("\nAll data processing complete!")

--- Stratified Split Complete ---
Test Pothole Files: ['both vibration.csv', 'right small (2).csv', 'left_medium_right.csv', 'right bad.csv']

Synthesizing 2000 scenarios for ../../data/combined/multi_class_pothole/train.csv...
Successfully saved to ../../data/combined/multi_class_pothole/train.csv
Synthesizing 500 scenarios for ../../data/combined/multi_class_pothole/test.csv...
Successfully saved to ../../data/combined/multi_class_pothole/test.csv

All data processing complete!
